In [88]:
import csv
import re


def parse_gff3_once(gff3_file):
    """
    Parse a GFF3 file once and store all relevant features in memory.

    Args:
        gff3_file (str): Path to the GFF3 file.

    Returns:
        dict: A dictionary where keys are gene symbols, and values are lists of CDS features.
    """
    data = {}

    with open(gff3_file, "r") as file:
        reader = csv.reader(file, delimiter="\t")

        for line in reader:
            # Skip header and comment lines
            if line[0].startswith("#") or len(line) < 9:
                continue

            # Unpack GFF3 columns
            seqid, source, feature_type, start, end, score, strand, phase, attributes = line

            # Only process CDS features
            if feature_type != "CDS":
                continue

            # Parse attributes into a dictionary
            attr_dict = {}
            for attr in attributes.split(";"):
                if "=" in attr:
                    key, value = attr.split("=", 1)
                    attr_dict[key.strip()] = value.strip()

            # Extract gene_name (or gene_id if gene_name is not available)
            gene_name = attr_dict.get("gene_name", None)
            if not gene_name:
                continue  # Skip entries without a gene_name

            # Initialize the gene entry in the dictionary if not already present
            if gene_name not in data:
                data[gene_name] = []

            # Append the CDS feature to the gene's list
            data[gene_name].append({
                "seqid": seqid,
                "start": int(start),
                "end": int(end),
                "strand": strand,
                "phase": int(phase) if phase in ("0", "1", "2") else None,
                "attributes": attr_dict
            })

    return data


def get_phase_from_preloaded_data(data, exon_start, exon_end, gene_symbol):
    """
    Retrieve the phase for a given pair of exon coordinates from preloaded GFF3 data.

    Args:
        data (dict): Preloaded GFF3 data (output of parse_gff3_once).
        exon_start (int): Start position of the exon (1-based).
        exon_end (int): End position of the exon (1-based).
        gene_symbol (str): Gene symbol to query.

    Returns:
        tuple: Phase (0, 1, 2), strand ('+', '-') for the given exon coordinates.

    Raises:
        ValueError: If the exon coordinates or the gene cannot be found.
    """
    # Check if the gene exists in the preloaded data
    if gene_symbol not in data:
        raise ValueError("Gene '{}' not found in the GFF3 data.".format(gene_symbol))

    # Search for the exon within the gene's CDS features
    for cds in data[gene_symbol]:
        if cds["start"] == exon_start and cds["end"] == exon_end:
            return cds["phase"], cds["strand"]

    # If no matching CDS entry is found
    raise ValueError(
        "Exon coordinates ({}, {}) not found within the CDS annotations for gene '{}'.".format(
            exon_start, exon_end, gene_symbol
        )
    )



In [60]:
gff3_file = "../source_data/gencode.v47.basic.annotation.gff3" 
gff3_data = parse_gff3_once(gff3_file)

In [61]:
import pandas as pd
df_info = pd.read_table('../source_data/EVENT_INFO-hg38.tab')

In [62]:
df_event = pd.read_csv('../outputs/neuron_up_inframe_miniexon_conserved.csv')

In [63]:
aa_seq = []
for event in df_event['EVENT']:
    df_select = df_info[df_info['EVENT'] == event]
    exon_start = int(df_select['CO_C1'].iloc[0].split(':')[1].split('-')[0])  # Example start coordinate
    exon_end = int(df_select['CO_C1'].iloc[0].split(':')[1].split('-')[1])  # Example end coordinate
    gene_symbol = df_select['GENE'].iloc[0]  # Gene symbol to filter
    try:
        # Step 2: Query the preloaded data for the phase
        phase, strand = get_phase_from_preloaded_data(gff3_data, exon_start, exon_end, gene_symbol)
        c1 = df_select['Seq_C1'].iloc[0]
        A = df_select['Seq_A'].iloc[0]
        from Bio.Seq import Seq
        if (abs(exon_end-exon_start)+1-phase) % 3 != 0:
            dna = Seq(c1[-((abs(exon_end-exon_start)+1-phase) % 3):] + A)
        else:
            dna = Seq(A)
        aa_seq.append(str(dna.translate()))
    except ValueError as e:
        aa_seq.append('Nan')
        print("Error: {}".format(e))
    

/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an er

Error: Exon coordinates (183037992, 183038472) not found within the CDS annotations for gene 'NCKAP1'.
Error: Gene 'DOPEY1' not found in the GFF3 data.


/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an er

Error: Gene 'AC138969.4' not found in the GFF3 data.


/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an er

...

/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an er

Error: Exon coordinates (32254148, 32254394) not found within the CDS annotations for gene 'FRY'.


/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an er

Error: Exon coordinates (224787022, 224787118) not found within the CDS annotations for gene 'DOCK10'.
Error: Exon coordinates (40857026, 40857159) not found within the CDS annotations for gene 'APBB2'.


/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/home/rltian/.local/lib/python3.12/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an er

In [64]:
df_event['exon_aa_seq'] = aa_seq

/tmp/ipykernel_2686905/3201834030.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_event['exon_aa_seq'] = aa_seq


In [110]:
df_protein = pd.read_table('../source_data/PROT_ISOFORMS-hg38.tab')
df_include = df_protein[df_protein['IsoformType']=='Incl_main']

In [111]:
df_add = df_protein[df_protein['EventID'].isin(rest)]

In [115]:
df_map = pd.concat([df_include, df_add],axis=0, ignore_index=True)

In [116]:
df_merge = df_event.merge(df_map, left_on = 'EVENT', right_on = 'EventID', how = 'left')

In [119]:
df_merge

,Unnamed: 0,GENE,EVENT,COORD,LENGTH,Whole_Brain_b,Cortex,Frontal_Gyrus_young,Frontal_Gyrus_old,Sup_Temporal_Gyrus,...,Heart_c,Muscle_b,Muscle_d,Muscle_e,ratio,exon_aa_seq,EventID,IsoformType,IsoformID,Ass
0,171,CACNA1G,HsaEX0011980,chr17:50594993-50595061,69,56.52,81.08,81.98,85.71,74.55,...,NaN,NaN,100.00,NaN,5.089609,EISKREDASGQLSCIQLPVDSQG,HsaEX0011980,Incl_main,ENSP00000352011,hg38
1,541,ANK1,HsaEX0004117,chr8:41700431-41700454,24,89.63,100.00,100.00,95.37,85.00,...,39.59,0.00,16.27,23.78,14.528156,GTAHITIM,HsaEX0004117,Incl_main,ENSP00000265709,hg38
2,585,APBA2,HsaEX0005036,chr15:29094278-29094313,36,61.10,62.09,86.85,86.19,90.53,...,NaN,NaN,0.00,0.00,17.515541,RMQKAAKIKKKA,HsaEX0005036,Incl_main,ENSP00000453144,hg38
3,1047,NCKAP1,HsaEX0042009,chr2:183024978-183024995,18,75.85,74.32,62.26,75.76,90.52,...,0.00,0.00,0.00,0.00,15.388893,Nan,HsaEX0042009,Incl_main,ENSP00000354251,hg38
4,1073,LIMCH1,HsaEX0035813,chr4:41687840-41687917,78,35.86,33.02,51.94,49.88,54.93,...,67.90,94.19,85.08,89.58,38.502870,KKSPREHFQAGPFSPCSPTPPGQSPN,HsaEX0035813,Incl_main,ENSP00000425631,hg38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230,167594,EWSR1,HsaEX1013573,chr22:29274268-29274282,15,100.00,100.00,100.00,100.00,100.00,...,NaN,NaN,67.34,100.00,5.031996,EGTST,HsaEX1013573,Incl_main,ENSP00000405947,hg38
231,167596,MTA1,HsaEX0040386,chr14:105468338-105468349,12,65.47,55.83,76.92,65.46,83.72,...,0.62,0.00,0.00,0.00,13.150036,GTYL,HsaEX0040386,Incl_main,ENSP00000394106,hg38
232,167705,RELN,HsaEX0053031,chr7:103478389-103478394,6,70.57,60.39,93.91,91.23,92.35,...,0.00,0.00,0.00,0.00,20.096071,LV,HsaEX0053031,Incl_main,ENSP00000392423,hg38
233,477724,TPD52,HsaEX0066655,chr8:80050445-80050471,27,23.14,73.74,72.29,80.52,84.01,...,24.09,0.00,11.71,7.75,33.963134,KLQAFSHSF,HsaEX0066655,Incl_main,ENSP00000429351,hg38


In [120]:
from Bio import SeqIO
import requests
from io import StringIO
import numpy as np
def get_protein_sequence(ensp_id):
    """Get protein sequence from main Ensembl REST API"""
    server = "https://rest.ensembl.org"  # Use main server
    ext = f"/sequence/id/{ensp_id}?type=protein"
    headers = {"Content-Type": "text/x-fasta"}
    
    response = requests.get(server + ext, headers=headers)
    if not response.ok:
        print(f"Error {response.status_code}: {response.text}")
        return None
    
    # Parse FASTA using Biopython
    fasta_string = StringIO(response.text)
    record = SeqIO.read(fasta_string, "fasta")
    
    return record

# Test with working example
seq = []
for ensp_id in df_merge['IsoformID']:
    seq_record = get_protein_sequence(ensp_id)

    if seq_record:
        seq.append((ensp_id, str(seq_record.seq)))
    else:
        seq.append((ensp_id, np.nan))

Error 400: {"error":"ID 'ENST00000278187fB2881' not found"}
Error 400: {"error":"ID 'nan' not found"}
Error 400: {"error":"ID 'ENST00000511685fB4855' not found"}
Error 400: {"error":"ID 'ENST00000542859fB2823' not found"}
Error 400: {"error":"ID 'ENST00000361580fB3734' not found"}
Error 400: {"error":"ID 'ENST00000356362fB5632' not found"}
Error 400: {"error":"ID 'ENST00000265997fB1791' not found"}
Error 400: {"error":"ID 'ENSP00000238855' not found"}
Error 400: {"error":"ID 'ENSP00000261517' not found"}
Error 400: {"error":"ID 'ENSP00000261517' not found"}
Error 400: {"error":"ID 'ENST00000397558fB1408' not found"}
Error 400: {"error":"ID 'nan' not found"}
Error 400: {"error":"ID 'ENSP00000409667' not found"}
Error 400: {"error":"ID 'ENSP00000409667' not found"}
Error 400: {"error":"ID 'ENST00000379643fB4094' not found"}
Error 400: {"error":"ID 'ENST00000366574fB6093' not found"}
Error 400: {"error":"ID 'ENST00000379516fB6600' not found"}
Error 400: {"error":"ID 'ENSP00000416400' not 

In [125]:
df_merge['aa_seq'] = pd.DataFrame(seq)[1]
df_merge.to_csv('../test/neuron_up_inframe_miniexon_conserved_aaseq.csv')

In [123]:
df_merge['protein_length'] = df_merge.apply(lambda x:len(x['aa_seq']), axis=1)

TypeError: object of type 'float' has no len()

In [124]:
df_merge

,Unnamed: 0,GENE,EVENT,COORD,LENGTH,Whole_Brain_b,Cortex,Frontal_Gyrus_young,Frontal_Gyrus_old,Sup_Temporal_Gyrus,...,Muscle_b,Muscle_d,Muscle_e,ratio,exon_aa_seq,EventID,IsoformType,IsoformID,Ass,aa_seq
0,171,CACNA1G,HsaEX0011980,chr17:50594993-50595061,69,56.52,81.08,81.98,85.71,74.55,...,NaN,100.00,NaN,5.089609,EISKREDASGQLSCIQLPVDSQG,HsaEX0011980,Incl_main,ENSP00000352011,hg38,MDEEEDGAGAEESGQPRSFMRLNDLSGAGGRPGPGSAEKDPGSADS...
1,541,ANK1,HsaEX0004117,chr8:41700431-41700454,24,89.63,100.00,100.00,95.37,85.00,...,0.00,16.27,23.78,14.528156,GTAHITIM,HsaEX0004117,Incl_main,ENSP00000265709,hg38,MAQAAKQLKKIKDIEAQALQEQKEKEESNRKRRNRSRDRKKKADAA...
2,585,APBA2,HsaEX0005036,chr15:29094278-29094313,36,61.10,62.09,86.85,86.19,90.53,...,NaN,0.00,0.00,17.515541,RMQKAAKIKKKA,HsaEX0005036,Incl_main,ENSP00000453144,hg38,MAHRKLESVGSGMLDHRVRPGPVPHSQEPESEDMELPLEGYVPEGL...
3,1047,NCKAP1,HsaEX0042009,chr2:183024978-183024995,18,75.85,74.32,62.26,75.76,90.52,...,0.00,0.00,0.00,15.388893,Nan,HsaEX0042009,Incl_main,ENSP00000354251,hg38,MSRSVLQPSQQKLAEKLTILNDRGVGMLTRLYNIKKQGQVWKACGD...
4,1073,LIMCH1,HsaEX0035813,chr4:41687840-41687917,78,35.86,33.02,51.94,49.88,54.93,...,94.19,85.08,89.58,38.502870,KKSPREHFQAGPFSPCSPTPPGQSPN,HsaEX0035813,Incl_main,ENSP00000425631,hg38,MRKDTDDIESPKRSIRDSGYIDCWDSERSDSLSPPRHGRDDSFDSL...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230,167594,EWSR1,HsaEX1013573,chr22:29274268-29274282,15,100.00,100.00,100.00,100.00,100.00,...,NaN,67.34,100.00,5.031996,EGTST,HsaEX1013573,Incl_main,ENSP00000405947,hg38,MASTDYSTYSQAAAQQGYSAYTAQPTQGYAQTTQQAYGQQSYGTYG...
231,167596,MTA1,HsaEX0040386,chr14:105468338-105468349,12,65.47,55.83,76.92,65.46,83.72,...,0.00,0.00,0.00,13.150036,GTYL,HsaEX0040386,Incl_main,ENSP00000394106,hg38,QIDQFLVVARSVGTFARALDCSSSVRQPSLHMSAAAASRDITLFHA...
232,167705,RELN,HsaEX0053031,chr7:103478389-103478394,6,70.57,60.39,93.91,91.23,92.35,...,0.00,0.00,0.00,20.096071,LV,HsaEX0053031,Incl_main,ENSP00000392423,hg38,MERSGWARQTFLLALLLGATLRARAAAGYYPRFSPFFFLCTHHGEL...
233,477724,TPD52,HsaEX0066655,chr8:80050445-80050471,27,23.14,73.74,72.29,80.52,84.01,...,0.00,11.71,7.75,33.963134,KLQAFSHSF,HsaEX0066655,Incl_main,ENSP00000429351,hg38,MDCREMDLYEDYQSPFDFDAGVNKSYLYLSPSGNSSPPGSPTLQKF...


In [53]:
df = df.sort_values('protein_length')

In [55]:
df.to_csv('./neuron_up_inframe_miniexon_conserved_aaseq.csv')